# 04. Rantai E1: apakah lapisan Groq dan detektor berbayar layak?

Skor indikasi AI di produksi punya tiga lapis, dan tanpa notebook ini hanya satu yang pernah diukur
(heuristik, notebook 02-03):

```
detektor eksternal (Winston)   ← bagian B, berbayar per KATA
  └─ ai_probability Groq       ← bagian A, BELUM PERNAH DIUKUR sebelum notebook ini
       └─ heuristik ai_score   ← notebook 02-03, terukur
```

Lapisan tengah menanggung seluruh beban begitu kredit detektor habis, dan itu bukan kemungkinan yang jauh.
Selama ini ia dianggap lebih baik daripada heuristik tanpa satu pun angka. Kalau ternyata tidak, jaring
pengaman yang diasumsikan sebenarnya tidak ada.

**Aturan keras notebook ini:** kedua mesin dinilai pada **subset yang sama persis**. Membandingkan angka
satu mesin di 40 sampel dengan angka heuristik di 999 sampel adalah perbandingan yang tidak sah.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from src import config

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 140)
config.ensure_dirs()
GOLD = config.GOLD_DIR / "gold_set.csv"

In [2]:
import os

from src import evaluate_baseline as eb
from src import evaluate_detector as ed
from src import evaluate_groq as eg
from src import evaluation as ev
from src import generate, metrics, viz

eb.bootstrap_backend()
HAS_GOLD = GOLD.exists()

## Mekanisme yang dijaga (dapat dicoba tanpa kunci)

**1. Sampel seimbang dan berselang-seling.** Gold set tersimpan manusia dulu baru AI, dan kedua evaluasi
pasti berhenti di tengah jalan karena kredit atau kuota. Dengan urutan asli, potongan yang terkumpul
satu kelas saja, dan ROC-AUC tidak terdefinisi (pernah terjadi: 26 skor pertama ternyata 25 manusia dan 1 AI).

In [3]:
toy = [{"binary_label": 0, "text": f"m{i}"} for i in range(6)] + [{"binary_label": 1, "text": f"a{i}"} for i in range(6)]
urutan = [("M" if r["binary_label"] == 0 else "A") for r in ev.balanced_sample(toy, 8)]
print("urutan proses:", " ".join(urutan), "→ potongan berapa pun dari awal tetap kira-kira seimbang")

urutan proses: M A M A M A M A → potongan berapa pun dari awal tetap kira-kira seimbang


**2. Kunci cache memuat nama model dan versi.** Tanpa itu, mengganti `GROQ_MODEL` membaca skor model lama
dan melaporkannya sebagai hasil model baru.

In [4]:
k = ev.ScoreCache.key
pd.Series(
    {
        "model A, teks sama": k("model-a", "teks")[:12],
        "model B, teks sama": k("model-b", "teks")[:12],
        "model A, teks lain": k("model-a", "teks lain")[:12],
    },
    name="kunci cache (12 karakter pertama)",
).to_frame()

,kunci cache (12 karakter pertama)
"model A, teks sama",50d4e903e391
"model B, teks sama",da2eb890b260
"model A, teks lain",68301416d0b4


**3. Dua jenis 429 Groq yang penanganannya berlawanan.** Batas **per menit** pulih dalam hitungan detik dan
layak ditunggu; batas **per hari** tidak akan pulih berapa kali pun diulang. Keduanya terlihat identik bila
yang dibaca hanya kode status, jadi isi pesan yang dibaca.

In [5]:
pd.Series(
    {
        "per menit (tunggu, backoff)": generate.is_daily_quota('{"error":{"message":"Rate limit reached ... tokens per minute (TPM)"}}'),
        "per hari (berhenti seketika)": generate.is_daily_quota('{"error":{"message":"Rate limit reached ... tokens per day (TPD)"}}'),
    },
    name="dikenali sebagai jatah harian?",
).to_frame()

,dikenali sebagai jatah harian?
"per menit (tunggu, backoff)",False
per hari (berhenti seketika),True


## Bagian A. Lapisan Groq

Fungsi produksi **dipanggil langsung** (prompt, model, temperatur, dan pembatasan nilai dari
`backend/apps/analysis/providers/groq.py`), bukan disalin. Kegagalan sengaja **berisik**: klien produksi
meratakan semua kegagalan menjadi fallback supaya pengumpulan tugas tidak pernah gagal, tetapi kalibrasi yang
diam-diam melewati separuh sampel melaporkan angka yang salah dengan penuh percaya diri.

Jatah gratis ±64 sampel per hari (≈1.550 token per penilaian, jatah 100.000 token/hari untuk model besar).
Subset diambil berselang-seling manusia dan AI, dan skor tersimpan di cache sehingga hari berikutnya
melanjutkan.

In [6]:
LIMIT_GROQ = 64      # 0 = seluruh gold set (belasan hari pada jatah gratis)
SLEEP_GROQ = 1.0
EDUCATION_LEVEL = eg.DEFAULT_EDUCATION_LEVEL   # satu nilai untuk semua sampel agar jenjang tidak ikut bergerak

if not (HAS_GOLD and config.GROQ_API_KEY):
    print("Butuh gold_set.csv dan GROQ_API_KEY. Bagian A dilewati.")
    groq_ok = False
else:
    llm = eg.production_llm()
    model = os.getenv("GROQ_MODEL", llm.DEFAULT_MODEL)
    subset = ev.balanced_sample(eb.load_rows(GOLD, None, "exclude"), LIMIT_GROQ)
    print(f"Menilai {len(subset)} teks dengan {model}")
    groq_scores, heur_groq, labels_groq = eg.collect_scores(subset, EDUCATION_LEVEL, SLEEP_GROQ, model, llm)
    groq_ok = True

Butuh gold_set.csv dan GROQ_API_KEY. Bagian A dilewati.


In [7]:
if groq_ok:
    ev.print_engine_comparison(
        "Groq", groq_scores, heur_groq, labels_groq,
        subheading=f"Model yang diukur: {model}",
        worse_note="  Groq TIDAK lebih baik daripada heuristik pada subset ini: lapisan kedua bukan jaring pengaman.",
    )
    viz.plot_roc({"Groq": (groq_scores, labels_groq), "heuristik": (heur_groq, labels_groq)}, title="Subset yang sama")
    plt.show()

> **Mengapa angka Groq di sini condong pesimis.** Prompt produksi meminta model menilai *jawaban mahasiswa*,
> sedangkan yang diberikan abstrak jurnal yang ditulis peneliti terlatih. Tulisan rapi cenderung dinilai
> lebih mirip AI.

## Bagian B. Detektor berbayar (Winston AI)

Winston mencantumkan `id` di daftar bahasa API-nya, tetapi itu klaim penyedia. Integrasi pertama memakai
**Sapling** dan dibatalkan sebelum sempat diukur setelah ketahuan **English-only**, padahal produk menilai
esai Indonesia. Bagian ini mengubah klaim menjadi angka.

Tiga hal yang dijaga:

* **Kredit terpakai per kata** (pendaftaran baru memberi 2.500 kredit; gold set penuh ±170.000 kata).
  Karena itu limit awal kecil, sampel diseimbangkan, dan setiap skor yang pernah dibayar disimpan di cache.
* **Arah skornya terbalik.** Winston mengembalikan *human score* (0 = hampir pasti AI, 100 = manusia).
  Yang disimpan selalu probabilitas AI yang sudah dibalik: `1 - human/100`, tepat sekali, di
  `providers/winston.py`. Dibuang atau digandakan, mahasiswa jujur mendapat skor AI tertinggi dan angkanya
  tetap tampak masuk akal.
* **Teks di luar rentang panjang dibuang lebih dulu** (minimal 300 karakter), supaya sampel tidak diam-diam
  condong ke teks panjang di tengah proses.

In [8]:
pd.DataFrame(
    {"human score Winston": [95, 50, 5]},
).assign(**{"probabilitas AI (1 - human/100)": lambda d: 1 - d["human score Winston"] / 100})

,human score Winston,probabilitas AI (1 - human/100)
0,95,0.05
1,50,0.50
2,5,0.95


In [9]:
LIMIT_DETEKTOR = 40    # kecil dengan sengaja: ±120 kata per sampel
SLEEP_DETEKTOR = 1.0

if not (HAS_GOLD and config.detector_api_key()):
    print("Butuh gold_set.csv dan WINSTON_API_KEY. Bagian B dilewati.")
    det_ok = False
else:
    from apps.analysis.providers import winston as detector

    rows_all, dilewati = ed.eligible_rows(eb.load_rows(GOLD, None, "exclude"), detector)
    subset = ev.balanced_sample(rows_all, LIMIT_DETEKTOR)
    kata = sum(len(r["text"].split()) for r in subset)
    print(f"{dilewati} teks dilewati (panjang di luar rentang); menilai {len(subset)} teks, {kata} kata (≈ {kata} kredit bila semua baru)")
    det_scores, heur_det, labels_det = ed.collect_scores(subset, config.detector_api_key(), SLEEP_DETEKTOR, detector)
    det_ok = True

Butuh gold_set.csv dan WINSTON_API_KEY. Bagian B dilewati.


In [10]:
if det_ok:
    ev.print_engine_comparison(
        detector.PROVIDER_NAME, det_scores, heur_det, labels_det,
        subheading=f"Versi model yang diukur: {detector.MODEL_VERSION}",
        worse_note="  Detektor TIDAK lebih baik: jangan menggeser ambang; pertimbangkan mencabut integrasinya.",
    )
    viz.plot_roc({detector.PROVIDER_NAME: (det_scores, labels_det), "heuristik": (heur_det, labels_det)}, title="Subset yang sama")
    plt.show()

## Keputusan

| Hasil | Tindakan |
|---|---|
| lapisan **tidak** mengungguli heuristik pada subset yang sama | **cabut** lapisannya, jangan menggeser ambang sampai angkanya bagus |
| mengungguli tipis (< 0,05 AUC) pada sampel kecil | perbesar `LIMIT_*` sebelum memutuskan |
| mengungguli jelas | ambang "FPR < 5%" baris detektor mengisi `MID_THRESHOLD` di `providers/winston.py`; **jangan** menyalin ambang heuristik, bentuk sebaran skornya berbeda |

Kerjakan bagian A **sebelum** menambah kredit detektor: pengukurannya gratis, dan hasilnya menentukan apakah
membayar detektor masih masuk akal.